In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df= pd.read_csv('qoute_dataset.csv')

In [8]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [9]:
df.shape

(3038, 2)

In [10]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [11]:
quotes= df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [12]:
import string

# Re-initialize quotes from the original DataFrame column to ensure a clean start
quotes = df['quote'].copy()

# Convert to lowercase
quotes = quotes.str.lower()

# Remove punctuation
translation_table = str.maketrans('', '', string.punctuation)
quotes = quotes.str.translate(translation_table)

# Display the cleaned quotes
display(quotes.head())

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [13]:
quotes= quotes.str.lower()


In [14]:
import string
translation_table = str.maketrans('', '', string.punctuation)
quotes = quotes.str.translate(translation_table)

In [15]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [17]:
vocab_size=10000
tokenizer= Tokenizer(num_words=vocab_size,)
tokenizer.fit_on_texts(quotes)

In [18]:
word_index= tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [19]:
sequence= tokenizer.texts_to_sequences(quotes)

In [20]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [21]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [22]:
X=[]
y=[]
for seq in sequence:
  for i in range(1,len(seq)):
    input_seq= seq[:i]
    output_seq= seq[i]
    X.append(seq[:i])
    y.append(seq[i])

In [23]:
len(X)

85271

In [24]:
len(y)

85271

In [25]:
max_len= max(len(x) for x in X)
print(max_len)

745


In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded= pad_sequences(maxlen= max_len,padding='pre',sequences=X)

In [27]:
y= np.array(y)

In [28]:
X_padded.shape

(85271, 745)

In [29]:
from tensorflow.keras.utils import to_categorical
y_one_hot= to_categorical(y,num_classes=vocab_size)

In [30]:
y.shape

(85271,)

In [31]:
y_one_hot.shape

(85271, 10000)

In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN,LSTM,Embedding

In [33]:
embed_dim= 50
rnn_units= 128

In [34]:
rnn_model= Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embed_dim,input_length=max_len))
rnn_model.add(
    SimpleRNN(units=rnn_units))
rnn_model.add(
    Dense(units=vocab_size,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [35]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
lstm_model= Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embed_dim,input_length=max_len))
lstm_model.add(
    LSTM(units=rnn_units))
lstm_model.add(
    Dense(units=vocab_size,activation='softmax'))


In [38]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
epochs= 10
batch_size=128

In [41]:
history_rnn=rnn_model.fit(
    x=X_padded,
    y=y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 69ms/step - accuracy: 0.0438 - loss: 6.7487 - val_accuracy: 0.0605 - val_loss: 6.5612
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0755 - loss: 6.1423 - val_accuracy: 0.0864 - val_loss: 6.3138
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1028 - loss: 5.7703 - val_accuracy: 0.1018 - val_loss: 6.2719
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1183 - loss: 5.4753 - val_accuracy: 0.1060 - val_loss: 6.2972
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1314 - loss: 5.2172 - val_accuracy: 0.1119 - val_loss: 6.3668
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1428 - loss: 4.9800 - val_accuracy: 0.1123 - val_loss: 6.4184
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1569 - loss: 4.7584 - val_accuracy: 0.1089 - val_loss: 6.4803
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1737 - loss: 4.5512 - 

In [42]:
epochs= 100
batch_size= 128
history_lstm= lstm_model.fit(
    x=X_padded,
    y=y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.0392 - loss: 6.7540 - val_accuracy: 0.0451 - val_loss: 6.6689
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 34s 56ms/step - accuracy: 0.0586 - loss: 6.3122 - val_accuracy: 0.0614 - val_loss: 6.5735
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0787 - loss: 6.0753 - val_accuracy: 0.0865 - val_loss: 6.4581
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 56ms/step - accuracy: 0.0959 - loss: 5.8536 - val_accuracy: 0.0942 - val_loss: 6.4375
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.1069 - loss: 5.6737 - val_accuracy: 0.1017 - val_loss: 6.4243
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.1178 - loss: 5.5069 - val_accuracy: 0.1025 - val_loss: 6.4483
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 34s 57ms/step - accuracy: 0.1245 - loss: 5.3509 - val_accuracy: 0.1061 - val_loss: 6.4440
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 34s 57ms/step - accuracy: 0.1331 - loss: 5

In [47]:
lstm_model.save('lstm_model.h5')

In [48]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [93]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word


In [50]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [99]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [100]:
seed_text = "life is"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

the


In [97]:
def generate_text(model,tokenizer,max_len,seed_text,n_words):
  for _ in range(n_words):
    next_word= predictor(model,tokenizer,seed_text,max_len)
    if next_word== "":
       break
    seed_text += ' ' + next_word
  return seed_text

In [101]:
seed_text= 'are you a'
generate_text= generate_text(lstm_model,tokenizer,max_len,seed_text,10)
print(generate_text)

are you a beautiful day but i think you can conquer fear upon


In [106]:
import pickle
with open('tokenizer.pkl','wb') as f:
  pickle.dump(tokenizer,f)

In [107]:
with open('max_len.pkl','wb')as f:
  pickle.dump(max_len,f)